# 🌐 Global AI Layoffs — Interactive Dashboard
Built with Dash + Plotly Express

**How to use:** Run all cells → click the link that appears → dashboard opens in your browser

## Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output
import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded ✅")

Libraries loaded ✅


## Load Data

In [2]:
df = pd.read_csv("../data/cleaned_layoff_dataset.csv", parse_dates=["date"])
df['year'] = df['date'].dt.year
df['quarter'] = df['date'].dt.quarter
df['year_quarter'] = df['date'].dt.to_period("Q").astype(str)

print(f"Data loaded ✅ — {df.shape[0]:,} rows")
df.head(3)

Data loaded ✅ — 2,465 rows


,company,location,layoff_count,date,pct_workforce,industry,stage,raised_mm,country,is_ai_company,year,month,quarter,year_quarter
0,Panda Squad,Sf Bay Area,6.0,2020-03-13,75.0,Consumer,Seed,1.0,US,False,2020,3,1,2020Q1
1,Hopskipdrive,Los Angeles,8.0,2020-03-13,10.0,Transportat…,Unknown,45.0,US,False,2020,3,1,2020Q1
2,Help.Com,Austin,16.0,2020-03-16,100.0,Support,Seed,6.0,US,False,2020,3,1,2020Q1


## Filter Options

In [3]:
years      = sorted(df['year'].dropna().unique().astype(int).tolist())
industries = ['All'] + sorted(df['industry'].dropna().unique().tolist())
countries  = ['All'] + sorted(df['country'].dropna().unique().tolist())

print(f"Years: {years[0]} → {years[-1]}")
print(f"Industries: {len(industries)-1}")
print(f"Countries: {len(countries)-1}")

Years: 2020 → 2026
Industries: 30
Countries: 59


## App Layout

In [4]:
app = Dash(__name__)

app.layout = html.Div([

    # ── HEADER ──────────────────────────────────────────
    html.Div([
        html.H1("🌐 Global AI Layoffs Dashboard",
                style={'margin': '0', 'fontSize': '26px', 'color': '#ffffff'}),
        html.P("Interactive analysis of global tech layoffs (2020–Present)",
               style={'margin': '4px 0 0 0', 'color': '#cce8e9', 'fontSize': '13px'})
    ], style={
        'backgroundColor': '#01696f',
        'padding': '20px 30px',
        'marginBottom': '20px'
    }),

    # ── FILTERS ─────────────────────────────────────────
    html.Div([

        html.Div([
            html.Label("📅 Year Range", style={'fontWeight': 'bold', 'fontSize': '13px'}),
            dcc.RangeSlider(
                id='year-slider',
                min=years[0], max=years[-1],
                value=[years[0], years[-1]],
                marks={y: str(y) for y in years},
                step=1
            )
        ], style={'flex': '2', 'marginRight': '30px'}),

        html.Div([
            html.Label("🏭 Industry", style={'fontWeight': 'bold', 'fontSize': '13px'}),
            dcc.Dropdown(
                id='industry-dropdown',
                options=[{'label': i, 'value': i} for i in industries],
                value='All', clearable=False
            )
        ], style={'flex': '1', 'marginRight': '20px'}),

        html.Div([
            html.Label("🌍 Country", style={'fontWeight': 'bold', 'fontSize': '13px'}),
            dcc.Dropdown(
                id='country-dropdown',
                options=[{'label': c, 'value': c} for c in countries],
                value='All', clearable=False
            )
        ], style={'flex': '1', 'marginRight': '20px'}),

        html.Div([
            html.Label("🤖 Company Type", style={'fontWeight': 'bold', 'fontSize': '13px'}),
            dcc.Dropdown(
                id='ai-dropdown',
                options=[
                    {'label': 'All',         'value': 'All'},
                    {'label': 'AI Company',     'value': True},
                    {'label': 'Non-AI Company', 'value': False}
                ],
                value='All', clearable=False
            )
        ], style={'flex': '1'}),

    ], style={
        'display': 'flex',
        'alignItems': 'flex-end',
        'padding': '10px 30px 20px',
        'backgroundColor': '#f9f8f5',
        'borderBottom': '1px solid #dcd9d5'
    }),

    # ── KPI CARDS ────────────────────────────────────────
    html.Div([
        html.Div(id='kpi-total-layoffs',  style={'flex': '1', 'margin': '0 10px'}),
        html.Div(id='kpi-total-companies',style={'flex': '1', 'margin': '0 10px'}),
        html.Div(id='kpi-top-industry',   style={'flex': '1', 'margin': '0 10px'}),
        html.Div(id='kpi-top-country',    style={'flex': '1', 'margin': '0 10px'}),
    ], style={
        'display': 'flex',
        'padding': '20px 20px 0',
        'backgroundColor': '#f9f8f5'
    }),

    # ── CHARTS ROW 1 — Full Width Line Chart ─────────────
    html.Div([
        dcc.Graph(id='chart-time', style={'height': '350px'})
    ], style={'padding': '20px 30px 0', 'backgroundColor': '#f9f8f5'}),

    # ── CHARTS ROW 2 — Bar + Map ─────────────────────────
    html.Div([
        html.Div([
            dcc.Graph(id='chart-companies', style={'height': '380px'})
        ], style={'flex': '1', 'marginRight': '15px'}),
        html.Div([
            dcc.Graph(id='chart-map', style={'height': '380px'})
        ], style={'flex': '1'})
    ], style={
        'display': 'flex',
        'padding': '20px 30px 0',
        'backgroundColor': '#f9f8f5'
    }),

    # ── CHARTS ROW 3 — Bubble Chart ──────────────────────
    html.Div([
        dcc.Graph(id='chart-bubble', style={'height': '420px'})
    ], style={'padding': '20px 30px', 'backgroundColor': '#f9f8f5'}),

], style={'fontFamily': 'Inter, sans-serif', 'backgroundColor': '#f9f8f5'})

print("Layout defined ✅")

Layout defined ✅


## All Callbacks

In [5]:
def filter_df(year_range, industry, country, ai_flag):
    dff = df[
        (df['year'] >= year_range[0]) &
        (df['year'] <= year_range[1])
    ].copy()
    if industry != 'All':
        dff = dff[dff['industry'] == industry]
    if country != 'All':
        dff = dff[dff['country'] == country]
    if ai_flag != 'All':
        dff = dff[dff['is_ai_company'] == ai_flag]
    return dff


def make_kpi_card(title, value, icon):
    return html.Div([
        html.P(f"{icon} {title}",
               style={'margin': '0 0 6px 0', 'fontSize': '12px',
                      'color': '#7a7974', 'fontWeight': '600',
                      'textTransform': 'uppercase', 'letterSpacing': '0.5px'}),
        html.H3(value,
                style={'margin': '0', 'fontSize': '26px',
                       'color': '#01696f', 'fontWeight': '700'})
    ], style={
        'backgroundColor': '#ffffff',
        'padding': '18px 20px',
        'borderRadius': '10px',
        'boxShadow': '0 2px 8px rgba(0,0,0,0.07)',
        'borderLeft': '4px solid #01696f'
    })


# ── SINGLE CALLBACK FOR EVERYTHING ──────────────────────
@app.callback(
    Output('kpi-total-layoffs',   'children'),
    Output('kpi-total-companies', 'children'),
    Output('kpi-top-industry',    'children'),
    Output('kpi-top-country',     'children'),
    Output('chart-time',          'figure'),
    Output('chart-companies',     'figure'),
    Output('chart-map',           'figure'),
    Output('chart-bubble',        'figure'),
    Input('year-slider',          'value'),
    Input('industry-dropdown',    'value'),
    Input('country-dropdown',     'value'),
    Input('ai-dropdown',          'value'),
)
def update_dashboard(year_range, industry, country, ai_flag):
    dff = filter_df(year_range, industry, country, ai_flag)

    # ── KPI VALUES ───────────────────────────────────────
    total_layoffs   = f"{int(dff['layoff_count'].sum()):,}"
    total_companies = f"{dff['company'].nunique():,}"
    top_industry    = dff.groupby('industry')['layoff_count'].sum().idxmax() \
                      if not dff.empty else 'N/A'
    top_country     = dff.groupby('country')['layoff_count'].sum().idxmax() \
                      if not dff.empty else 'N/A'

    kpi1 = make_kpi_card("Total Layoffs",    total_layoffs,   "👥")
    kpi2 = make_kpi_card("Companies Affected", total_companies, "🏢")
    kpi3 = make_kpi_card("Top Industry",     top_industry,    "🏭")
    kpi4 = make_kpi_card("Top Country",      top_country,     "🌍")

    # ── CHART 1: Layoffs Over Time ───────────────────────
    time_df = dff.groupby('year_quarter')['layoff_count'].sum().reset_index()
    fig1 = px.line(
        time_df, x='year_quarter', y='layoff_count',
        title='📈 Layoffs Over Time',
        markers=True,
        color_discrete_sequence=['#01696f'],
        labels={'year_quarter': 'Quarter', 'layoff_count': 'Total Layoffs'}
    )
    fig1.update_layout(
        plot_bgcolor='#ffffff', paper_bgcolor='#ffffff',
        xaxis_tickangle=-45, margin=dict(t=40, b=40)
    )

    # ── CHART 2: Top 10 Companies ────────────────────────
    top_cos = (
        dff.groupby('company')['layoff_count']
        .sum().nlargest(10).reset_index()
    )
    fig2 = px.bar(
        top_cos, x='layoff_count', y='company',
        orientation='h',
        title='🏢 Top 10 Companies by Layoffs',
        color='layoff_count',
        color_continuous_scale='Teal',
        labels={'layoff_count': 'Total Layoffs', 'company': ''}
    )
    fig2.update_layout(
        yaxis={'categoryorder': 'total ascending'},
        plot_bgcolor='#ffffff', paper_bgcolor='#ffffff',
        coloraxis_showscale=False, margin=dict(t=40)
    )

    # ── CHART 3: World Map ───────────────────────────────
    map_df = dff.groupby('country')['layoff_count'].sum().reset_index()
    fig3 = px.choropleth(
        map_df, locations='country', locationmode='country names',
        color='layoff_count',
        title='🌍 Layoffs by Country',
        color_continuous_scale='Teal',
        labels={'layoff_count': 'Total Layoffs'}
    )
    fig3.update_layout(
        paper_bgcolor='#ffffff',
        margin=dict(t=40, b=0, l=0, r=0),
        coloraxis_showscale=False
    )

    # ── CHART 4: Bubble Chart ────────────────────────────
    stage_df = (
        dff.groupby('stage').agg(
            Total_layoffs  =('layoff_count', 'sum'),
            average_layoffs=('layoff_count', 'mean'),
            layoff_events  =('layoff_count', 'count')
        ).reset_index()
    )
    fig4 = px.scatter(
        stage_df,
        x='average_layoffs', y='layoff_events',
        size='Total_layoffs', color='stage',
        title='💰 Funding Stage Risk: Avg Layoffs vs Frequency',
        labels={
            'average_layoffs': 'Average Layoffs per Event',
            'layoff_events'  : 'Number of Events',
            'Total_layoffs'  : 'Total Layoffs'
        },
        hover_name='stage',
        size_max=70,
        text='stage',
        template='plotly_white'
    )
    fig4.update_traces(textposition='top center', mode='markers+text',
                       textfont=dict(size=10))
    fig4.update_layout(
        plot_bgcolor='#ffffff', paper_bgcolor='#ffffff',
        showlegend=False, margin=dict(t=40)
    )

    return kpi1, kpi2, kpi3, kpi4, fig1, fig2, fig3, fig4


print("Callbacks registered ✅")

Callbacks registered ✅


## Run the App

In [ ]:
if __name__ == '__main__':
    app.run(debug=True, port=8050)